# jaxfne — Étude No. 11 · Local Omission Task

Omission oddball: on most trials a standard tone plays at a fixed slot
(`expected`); on omission trials the same slot is silent (`omitted`). The
local omission mismatch response is the post-slot firing-rate difference
between the two — a prediction-violation signal that does not require any
actual sensory input on the omission trial itself.

Uses `jaxfne.omission_oddball_paradigm` (`jaxfne/paradigm.py`, already
implemented) directly, rather than hand-building `StimulusSchedule` events
as Étude 9/10 did — its `ParadigmCondition`s are natively accepted by
`simulate(paradigm=...)` (`Model._resolve_stimulus_schedule` converts them
via `stimulus_schedule()`).

## 1. Setup

In [1]:
import numpy as np
import jax.numpy as jnp
import jaxfne as jtfne

print("jaxfne", jtfne.__version__)


jaxfne 0.4.4


## 2. Config

In [2]:
N_NEURONS = 200
SEED = 0
DT_MS = 0.5
PRE_MS = 200.0          # pre_stimulus_buffer_ms -- fixation before the stimulus slot
STIM_MS = 100.0         # standard_duration_ms -- the stimulus/omission slot itself
POST_MS = 300.0         # post_stimulus_buffer_ms -- window after the slot
TOTAL_MS = PRE_MS + STIM_MS + POST_MS
N_TRIALS = 10           # trials per condition

cfg = (
    jtfne.build_laminar_column(name="V1", n=N_NEURONS, ei_profile="canonical")
    .runtime(seed=SEED, recurrent_backend="edge_list")
    .set_emitter("izhikevich", "cortical_eig")
    .probes(["spikes", "V_m"], n_contacts=8)
    .field(domain="laminar_column", conductivity="proxy", boundary="mean_zero_neumann")
)
model = jtfne.construct(cfg)
nt = model.neuron_table()
l4e_idx = [i for i, r in enumerate(nt) if r.get("layer") == "L4" and r.get("cell_type") == "E"]
print(f"L4 E neurons: {len(l4e_idx)} / {N_NEURONS}")


L4 E neurons: 14 / 200


## 3. Paradigm

`omission_oddball_paradigm` returns 3 conditions (`expected`, `unexpected`,
`omitted`), each a `ParadigmCondition` with 3 `ParadigmEvent`s: a
`trial_start` marker, the stimulus/omission slot itself, and a
`post_stimulus`/`post_omission` marker. This étude uses `expected` vs
`omitted` (the local-omission contrast); `unexpected` (a genuinely
different tone in the slot) is left for a future deviant-vs-omission étude.

CAVEAT (confirmed by direct inspection of `stimulus_schedule()`, not
assumed): its drive-event heuristic is `is_drive = not is_omission and
onset_ms is not None` — this makes the `trial_start`/`post_stimulus`
marker events ALSO inject the default drive amplitude (5.0), not just the
actual stimulus event. This affects `expected` and `omitted` identically
(both share the same marker events), so the paired local-omission contrast
below is still valid — but neither condition has a truly silent pre/post
window, which matters if you reuse these conditions for a different
contrast.

In [3]:
paradigm = jtfne.omission_oddball_paradigm(
    standard_onset_ms=PRE_MS, standard_duration_ms=STIM_MS,
    post_stimulus_buffer_ms=POST_MS, pre_stimulus_buffer_ms=PRE_MS,
)
cond_expected = next(c for c in paradigm.conditions if c.name == "expected")
cond_omitted = next(c for c in paradigm.conditions if c.name == "omitted")
print("conditions:", [c.name for c in paradigm.conditions])


conditions: ['expected', 'unexpected', 'omitted']


## 4. Run

In [4]:
def post_slot_rate(condition, trial_seed):
    window_start = int(PRE_MS / DT_MS)
    window_end = int((PRE_MS + STIM_MS + 100.0) / DT_MS)
    sig = jtfne.simulate(
        model, sim=jtfne.Simulation(duration_ms=TOTAL_MS, dt_ms=DT_MS, seed=trial_seed),
        paradigm=condition,
    )
    assert bool(jnp.all(jnp.isfinite(sig.V_m))), "non-finite V_m -- do not trust this run"
    spk = np.asarray(sig.spikes)
    return spk[window_start:window_end, l4e_idx].mean() * 1000.0 / DT_MS

rates_expected = np.array([post_slot_rate(cond_expected, SEED + i) for i in range(N_TRIALS)])
rates_omitted = np.array([post_slot_rate(cond_omitted, SEED + i) for i in range(N_TRIALS)])
print(f"expected: {rates_expected.mean():.2f}+-{rates_expected.std():.2f} Hz")
print(f"omitted:  {rates_omitted.mean():.2f}+-{rates_omitted.std():.2f} Hz")


expected: 14.93+-0.14 Hz
omitted:  10.00+-0.00 Hz


## 5. Objective — local omission mismatch response

In [5]:
omission_mismatch_hz = float(rates_expected.mean() - rates_omitted.mean())
print(f"local omission mismatch (expected - omitted): {omission_mismatch_hz:+.2f} Hz")

# Computational diagnostic, not a claim of biological omission-response/
# predictive-coding validation -- a same-model firing-rate comparison
# between a real-stimulus slot and a silent slot, not a validated ERP result.
assert np.isfinite(omission_mismatch_hz)


local omission mismatch (expected - omitted): +4.93 Hz


## 6. Export

In [6]:
import json as _json
from pathlib import Path

OUT_DIR = Path("local/etude11")
OUT_DIR.mkdir(parents=True, exist_ok=True)

manifest = {
    "notebook": "jaxfne_etude_no_11_omission_local",
    "jaxfne_version": jtfne.__version__,
    "paradigm": {
        "name": "omission_local", "pre_ms": PRE_MS, "stim_ms": STIM_MS, "post_ms": POST_MS,
        "stimulus_target": "L4_E_neurons", "n_stimulus_targets": len(l4e_idx),
    },
    "config": {"N_NEURONS": N_NEURONS, "N_TRIALS": N_TRIALS, "DT_MS": DT_MS, "SEED": SEED},
    "results": {
        "expected_rate_hz_mean": float(rates_expected.mean()), "expected_rate_hz_std": float(rates_expected.std()),
        "omitted_rate_hz_mean": float(rates_omitted.mean()), "omitted_rate_hz_std": float(rates_omitted.std()),
        "omission_mismatch_hz": omission_mismatch_hz,
    },
}
(OUT_DIR / "manifest.json").write_text(_json.dumps(manifest, indent=2))
print("wrote", OUT_DIR / "manifest.json")


wrote local/etude11/manifest.json
